In [2]:
import pandas as pd
import re
import os

from kalshi_client import KalshiClient
from polymarket_client import PolymarketClient
from utils import create_match_keys

In [3]:
kalshi = KalshiClient()
kalshi.fetch_data()
kalshi_markets, kalshi_events = kalshi.get_separated_dfs()

poly = PolymarketClient()
poly.fetch_events()
poly_markets, poly_events = poly.get_separated_dfs()

common_cols = list(set(poly_events.columns) & set(kalshi_events.columns))

poly_events = poly_events[common_cols]
kalshi_events = kalshi_events[common_cols]
kalshi_events

Fetching Kalshi events...
Fetching Kalshi markets...
Adjusting missing events to 500 to avoid excessive API calls.
Backfilling 500 missing events from Kalshi...


/Users/blakeuribe/Desktop/quasibets_testing/utils.py:67: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_dt = pd.to_datetime(df[col], utc=True, errors='coerce')
/Users/blakeuribe/Desktop/quasibets_testing/utils.py:67: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_dt = pd.to_datetime(df[col], utc=True, errors='coerce')


Fetching Polymarket events...
Processing Polymarket markets...


/Users/blakeuribe/Desktop/quasibets_testing/utils.py:67: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_dt = pd.to_datetime(df[col], utc=True, errors='coerce')


,event_id,platform,event_title
0,KXATPCHALLENGERMATCH-26MAR29ARNRAQ,kalshi,arnaboldi vs raquillet
2,KXATPCHALLENGERMATCH-26MAR29MONBAR,kalshi,montes de la torre vs barreto sanchez
4,KXATPCHALLENGERMATCH-26MAR29PAUPIE,kalshi,paulson vs pieri
6,KXMLBRFI-26MAR311907COLTOR,kalshi,colorado vs toronto first inning run
7,KXMLBRFI-26MAR311915ATHATL,kalshi,a s vs atlanta first inning run
...,...,...,...
3704,KXOWGAME-26MAR29ALALQ,kalshi,ocs emea stage 1 2026 anyone s legend vs al qa...
3712,KXNCAABBGAME-26MAR281300LALHCC,kalshi,lafayette vs holy cross
3714,KXNCAABBGAME-26MAR281300NJITBRY,kalshi,njit vs bryant
3716,KXATPCHALLENGERMATCH-26MAR28GOJFIC,kalshi,gojo vs ficovich


In [4]:

kalshi_events = create_match_keys(kalshi_events, ['event_title'])
poly_events = create_match_keys(poly_events, ['event_title'])

display(poly_events)
kalshi_events

,event_id,platform,event_title,match_key
0,100079,poly,paramount close warner bros acquisition by end...,2026_2026
1,100229,poly,close warner bros acquisition,unknown_9999
16,100252,poly,merab dvalishivili fight next,next_9999
56,100253,poly,tabi fdv above one day after launch,one_9999
60,100260,poly,petr yan fight next,next_petr_yan_9999
...,...,...,...,...
28526,99573,poly,netanyahu visit nyc by march 31,march_netanyahu_9999
28527,99578,poly,netanyahu arrested by march 31,march_netanyahu_9999
28528,99600,poly,eu dissolves before 2027,unknown_2027
28529,99601,poly,any country withdraws from eu before 2027,unknown_2027


,event_id,platform,event_title,match_key
0,KXATPCHALLENGERMATCH-26MAR29ARNRAQ,kalshi,arnaboldi vs raquillet,unknown_2026
2,KXATPCHALLENGERMATCH-26MAR29MONBAR,kalshi,montes de la torre vs barreto sanchez,de_la_2026
4,KXATPCHALLENGERMATCH-26MAR29PAUPIE,kalshi,paulson vs pieri,unknown_2026
6,KXMLBRFI-26MAR311907COLTOR,kalshi,colorado vs toronto first inning run,colorado_first_toronto_2026
7,KXMLBRFI-26MAR311915ATHATL,kalshi,a s vs atlanta first inning run,atlanta_first_2026
...,...,...,...,...
3704,KXOWGAME-26MAR29ALALQ,kalshi,ocs emea stage 1 2026 anyone s legend vs al qa...,2026_al_2026
3712,KXNCAABBGAME-26MAR281300LALHCC,kalshi,lafayette vs holy cross,holy_lafayette_2026
3714,KXNCAABBGAME-26MAR281300NJITBRY,kalshi,njit vs bryant,unknown_2026
3716,KXATPCHALLENGERMATCH-26MAR28GOJFIC,kalshi,gojo vs ficovich,unknown_2026


In [6]:
from sentence_transformers import SentenceTransformer, util
from thefuzz import fuzz

# Load once to save memory
model = SentenceTransformer('all-MiniLM-L6-v2')

def get_market_matches(df_kalshi, df_poly, threshold=0.60):
    """
    Ranks matches and returns a DataFrame where the IDs 
    are stored in a single dictionary column for easy hydration.
    """
    # 1. Clean and prepare titles
    k_titles = df_kalshi['event_title'].astype(str).tolist()
    p_titles = df_poly['event_title'].astype(str).tolist()
    
    if not k_titles or not p_titles:
        return pd.DataFrame()

    # 2. Vectorize and compute similarity
    k_embeddings = model.encode(k_titles, convert_to_tensor=True)
    p_embeddings = model.encode(p_titles, convert_to_tensor=True)
    cosine_scores = util.cos_sim(k_embeddings, p_embeddings)

    all_results = []
    for i, k_title in enumerate(k_titles):
        best_match_idx = cosine_scores[i].argmax().item()
        semantic_score = cosine_scores[i][best_match_idx].item()
        
        if semantic_score >= threshold:
            p_match_title = p_titles[best_match_idx]
            
            # Fuzzy secondary check
            fuzzy_val = fuzz.token_sort_ratio(k_title, p_match_title) / 100.0
            
            all_results.append({
                # The ID Dictionary Column
                'platform_ids': {
                    'kalshi': df_kalshi.iloc[i]['event_id'],
                    'poly': df_poly.iloc[best_match_idx]['event_id']
                },
                'kalshi_title': k_title,
                'poly_match': p_match_title,
                'total_score': round((semantic_score + fuzzy_val) / 2, 4),
                'keys': {
                    'kalshi': df_kalshi.iloc[i].get('match_key', 'N/A'),
                    'poly': df_poly.iloc[best_match_idx].get('match_key', 'N/A')
                },
            })

    if not all_results:
        return pd.DataFrame()
        
    # Create DF and sort by best match
    match_df = pd.DataFrame(all_results).sort_values(by='total_score', ascending=False)
    return match_df.reset_index(drop=True)

# Run it
match_df = get_market_matches(kalshi_events, poly_events)
match_df

/Users/blakeuribe/Desktop/quasibets_testing/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6891.04it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,platform_ids,kalshi_title,poly_match,total_score,keys
0,"{'kalshi': 'KXKHLGAME-26MAR311000SALAVT', 'pol...",salavat yulaev ufa vs avtomobilist yekaterinburg,khl salavat yulaev ufa vs avtomobilist yekater...,0.9490,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
1,"{'kalshi': 'KXKHLGAME-26MAR311230SKACSK', 'pol...",ska st petersburg vs cska moscow,khl ska st petersburg vs cska moscow,0.9454,"{'kalshi': 'moscow_petersburg_2026', 'poly': '..."
2,{'kalshi': 'KXATPCHALLENGERMATCH-26MAR29CARLLA...,carreno busta vs llamas ruiz,alicante pablo carreno busta vs pablo llamas ruiz,0.8266,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
3,"{'kalshi': 'KXMLBTB-26MAR281915KCATL', 'poly':...",kansas city vs atlanta total bases,kansas city royals vs atlanta braves,0.7943,"{'kalshi': 'atlanta_city_kansas_2026', 'poly':..."
4,"{'kalshi': 'KXDOTA2MAP-26MAR29XTREMETY-2', 'po...",esl one birmingham 2026 xtreme gaming vs team ...,dota 2 team yandex vs xtreme gaming bo3 esl on...,0.7898,"{'kalshi': '2026_birmingham_one_2026', 'poly':..."
...,...,...,...,...,...
316,"{'kalshi': 'KXWTAMATCH-26MAR29AKLTOM', 'poly':...",akli vs tomova,credit one charleston open qualification ayana...,0.4789,"{'kalshi': 'unknown_2026', 'poly': 'charleston..."
317,"{'kalshi': 'KXNCAABBGAME-26MAR281500STCVTH', '...",stanford vs virginia tech,connecticut huskies vs duke blue devils,0.4690,"{'kalshi': 'stanford_virginia_2026', 'poly': '..."
318,"{'kalshi': 'KXNHLGOAL-26MAR28PHIDET', 'poly': ...",phi flyers at det red wings player goals,bruins vs flyers,0.4664,"{'kalshi': 'unknown_2026', 'poly': 'bruins_9999'}"
319,"{'kalshi': 'KXNCAAWBTOTAL-26MAR29DUKEUCLA', 'p...",duke at ucla total points,connecticut huskies vs duke blue devils,0.4581,"{'kalshi': 'duke_2026', 'poly': 'blue_connecti..."


In [7]:
import pandas as pd
import os
import json

THRESHOLD = 0.80
filepath = 'data/arb_candidates.json'

new_matches_df = match_df[match_df['total_score'] > THRESHOLD]

if os.path.exists(filepath):
    existing_df = pd.read_json(filepath)
    combined_df = pd.concat([existing_df, new_matches_df]).drop_duplicates(subset=['platform_ids'])
else:
    combined_df = new_matches_df

combined_df.to_json(filepath, orient='records', indent=4)

display(combined_df.head())
# display(match_df[match_df['total_score'] > THRESHOLD])
# display(match_df.hist(column='total_score', bins=30))
# display(match_df['total_score'].describe())

,platform_ids,kalshi_title,poly_match,total_score,keys
0,"{'kalshi': 'KXKHLGAME-26MAR311000SALAVT', 'pol...",salavat yulaev ufa vs avtomobilist yekaterinburg,khl salavat yulaev ufa vs avtomobilist yekater...,0.9490,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
1,"{'kalshi': 'KXKHLGAME-26MAR311230SKACSK', 'pol...",ska st petersburg vs cska moscow,khl ska st petersburg vs cska moscow,0.9454,"{'kalshi': 'moscow_petersburg_2026', 'poly': '..."
2,{'kalshi': 'KXATPCHALLENGERMATCH-26MAR29CARLLA...,carreno busta vs llamas ruiz,alicante pablo carreno busta vs pablo llamas ruiz,0.8266,"{'kalshi': 'unknown_2026', 'poly': 'unknown_99..."
3,"{'kalshi': 'KXMLBHR-26MAR281610BOSCIN', 'poly'...",boston vs cincinnati home runs,boston red sox vs cincinnati reds,0.8065,"{'kalshi': 'boston_home_2026', 'poly': 'boston..."
4,"{'kalshi': 'KXLOLMAP-26MAR29TTWE-1', 'poly': '...",esports world cup china qualifier 2026 thunder...,lol team we vs thundertalk gaming bo3 esports ...,0.8423,"{'kalshi': 'china_2026', 'poly': 'china_9999'}"
